In [4]:
# load sciplex
import pandas as pd
cell_line_name = "A549"
deg_results = pd.read_parquet(f"../../data/raw/sciplex/{cell_line_name}_differential_expression_results.parquet")
deg_results = deg_results[~deg_results['condition'].str.startswith('plate')]

# Extract drug name using two patterns:
# 1. factor_valid_drug_names_DRUGNAME_dose_0_cellline
# 2. DRUGNAME_dose_0_cellline

import anndata

# add cell-line into condition
deg_results["condition"] = deg_results["condition"] + "_" + deg_results["cellline"]

# ──────────────────────────────────────────────────────
# 1) Pivot each metric so rows=condition, cols=gene
# ──────────────────────────────────────────────────────
metrics = {
    'logFC':     'logfc',
    'AveExpr':   'ave_expr',
    'P.Value':   'p_value',
    'adj.P.Val': 'adj_p_value',
    'B':         'b'
}

pivoted = {}
for orig_col, layer_name in metrics.items():
    pivoted[layer_name] = deg_results.pivot(
        index='condition',
        columns='gene',
        values=orig_col
    )

# capture the ordering
conditions = pivoted['logfc'].index
genes      = pivoted['logfc'].columns

# ──────────────────────────────────────────────────────
# 2) Build obs (one row per condition) and var (one row per gene)
# ──────────────────────────────────────────────────────
obs = (
    deg_results
    .drop_duplicates(subset=['condition'])
    .set_index('condition')
    [['cellline']]
    .loc[conditions]  # align order
)

# var with gene index and an explicit gene_name column
var = pd.DataFrame(index=genes)
var['gene_name'] = genes


In [5]:
# ──────────────────────────────────────────────────────
# 3) Create the AnnData, using logFC as X
# ──────────────────────────────────────────────────────
adata = anndata.AnnData(
    X   = pivoted['logfc'].loc[conditions, genes].values,
    obs = obs,
    var = var
)

# ──────────────────────────────────────────────────────
# 4) Stuff the other stats into layers
# ──────────────────────────────────────────────────────
for layer_name in ['ave_expr', 'p_value', 'adj_p_value', 'b']:
    adata.layers[layer_name] = (
        pivoted[layer_name]
        .loc[conditions, genes]
        .values
    )

adata.layers["logfc"] = adata.X
del adata.X

In [6]:
# First ensure the index is properly set
adata.obs = adata.obs.reset_index()
adata.obs['drug'] = adata.obs['condition'].str.extract(
    r'(?:factor_valid_drug_names_)?(.*?)_\d+_0_'
)[0]
adata.obs['dosage_uM'] = adata.obs.condition.str.extract(r'_(\d+)_0_[^_]+$')[0].astype(float) / 1000
adata.obs = adata.obs.set_index('condition')

In [10]:
import pandas as pd
import mygene

# assume df is your DataFrame, with df['gene'] containing the Ensembl IDs
df = adata.var

mg = mygene.MyGeneInfo()

# split human vs mouse (so we query with the right species)
human_ids = [i for i in df['gene_name'] if i.startswith('ENSG')]
mouse_ids = [i for i in df['gene_name'] if i.startswith('ENSMUSG')]

# batch‐query human symbols
human_q = mg.querymany(human_ids,
                       scopes='ensembl.gene',
                       fields='symbol',
                       species='human',
                       as_dataframe=True)

# batch‐query mouse symbols
mouse_q = mg.querymany(mouse_ids,
                       scopes='ensembl.gene',
                       fields='symbol',
                       species='mouse',
                       as_dataframe=True)

# combine results into one mapping dict: {ensembl_id: symbol}
mapping = {}
mapping.update(human_q['symbol'].to_dict())
mapping.update(mouse_q['symbol'].to_dict())

# add your new column
df['gene_symbol'] = df['gene_name'].map(mapping)

Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
2 input query terms found dup hits:	[('ENSG00000215156', 2), ('ENSG00000249738', 2)]
102 input query terms found no hit:	['ENSG00000112096', 'ENSG00000130489', 'ENSG00000130723', 'ENSG00000131484', 'ENSG00000132832', 'ENS
Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed
3 input query terms found no hit:	['ENSMUSG00000048406', 'ENSMUSG00000092093', 'ENSMUSG00000097542']


In [25]:
experiment_dict = dict(zip(
    pd.read_csv("../Sciplex/Experiment_dict_Sciplex.csv")['Unnamed: 0'],
    pd.read_csv("../Sciplex/Experiment_dict_Sciplex.csv")['0']
))

In [17]:
adata.obs['condition_base'] = adata.obs.index.str.replace(r'_[^_]+$', '', regex=True)

In [26]:
adata.obs['drug_name'] = adata.obs['condition_base'].map(experiment_dict)

In [31]:
adata.obs.drug_name = adata.obs.drug_name.str.replace(r'_[^_]+$', '', regex=True)

In [36]:
adata.write_h5ad("../../data/degs/sciplex3_a549_deg_adata.h5ad", compression="gzip")